In [1]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install -q openai-whisper flask flask-cors pyngrok \
  torch torchvision torchaudio indic-transliteration \
  numpy scipy soundfile deep-translator

!pip install -q --upgrade pyngrok

print("✅ All dependencies installed!")

✅ All dependencies installed!


In [2]:
# ============================================================
# CELL 2: Authenticate Ngrok
# ============================================================
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3ClOAinMTPnbr0CMjfHd7tOHPVs_4A2moPSWrLqixicxDnWo7"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ Ngrok authenticated!")

✅ Ngrok authenticated!


In [3]:
# ============================================================
# CELL 3: Imports & Config
# ============================================================
import os, io, json, logging, warnings, tempfile, threading, re, time
import numpy as np
import torch
import whisper
import soundfile as sf

from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
from deep_translator import GoogleTranslator

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

try:
    _t = GoogleTranslator(source="auto", target="ne").translate("test")
    print(f"✅ deep-translator working!")
except Exception as e:
    print(f"❌ deep-translator failed: {e}")

✅ Device: cuda
✅ GPU: Tesla T4
✅ deep-translator working!


In [4]:
# ============================================================
# CELL 4: Load Whisper
# ============================================================
print("⏳ Loading Whisper Large-v3...")

whisper_model = whisper.load_model(
    "large-v3",
    device=DEVICE,
    download_root="/content/models/whisper"
)

print("✅ Whisper Large-v3 loaded!")

⏳ Loading Whisper Large-v3...
✅ Whisper Large-v3 loaded!


In [5]:
# ============================================================
# CELL 5: Skip — No Gemini needed
# ============================================================
print("✅ Using deep-translator — no API key needed!")

✅ Using deep-translator — no API key needed!


In [4]:
# ============================================================
# CELL 6: Complete NLP Pipeline — STRICT & RULE-BASED
# ============================================================
import re, logging, os, tempfile, time

try:
    from indic_transliteration import sanscript
    from indic_transliteration.sanscript import transliterate
except Exception:
    sanscript = None
    transliterate = None

try:
    from deep_translator import GoogleTranslator
except Exception:
    GoogleTranslator = None

logger = logging.getLogger(__name__)

_DEVANAGARI_PATTERN = re.compile(r"[\u0900-\u097F]")
_BENGALI_PATTERN = re.compile(r"[\u0980-\u09FF]")
_LATIN_PATTERN = re.compile(r"[A-Za-z]")


def _normalize_spaces(text):
    return re.sub(r"\s+", " ", (text or "")).strip()


def _contains_latin(text):
    return bool(_LATIN_PATTERN.search(text or ""))


def _contains_devanagari(text):
    return bool(_DEVANAGARI_PATTERN.search(text or ""))


def _contains_bengali(text):
    return bool(_BENGALI_PATTERN.search(text or ""))


def _sanitize_punctuation_nepali(text):
    text = _normalize_spaces(text)
    text = re.sub(r"[\s\|\.]+$", "", text)
    if text and not text.endswith(("।", "?", "!")):
        text = text + "।"
    return text


def _devanagari_ratio(text):
    if not text:
        return 0.0
    letters = [ch for ch in text if ch.isalpha() or "\u0900" <= ch <= "\u097F"]
    if not letters:
        return 0.0
    devanagari = [ch for ch in letters if "\u0900" <= ch <= "\u097F"]
    return len(devanagari) / len(letters)


def _is_valid_nepali_output(text):
    if not text or not text.strip():
        return False
    if _contains_latin(text):
        return False
    if not _contains_devanagari(text):
        return False
    return _devanagari_ratio(text) >= 0.6


# ══════════════════════════════════════════════════════════════
# STRICT PHRASE RULES
# ══════════════════════════════════════════════════════════════
STRICT_PHRASE_RULES = [
    (r"^(?:my\s+)?atm\s+card\s+is\s+not\s+working(?:\s+please\s+help)?$", "मेरो एटीएम कार्ड काम गरिरहेको छैन।"),
    (r"^(?:my\s+)?debit\s+card\s+is\s+not\s+working(?:\s+please\s+help)?$", "मेरो डेबिट कार्ड काम गरिरहेको छैन।"),
    (r"^(?:my\s+)?credit\s+card\s+is\s+not\s+working(?:\s+please\s+help)?$", "मेरो क्रेडिट कार्ड काम गरिरहेको छैन।"),
    (r"^please\s+block\s+my\s+card$", "कृपया मेरो कार्ड ब्लक गर्नुहोस्।"),
    (r"^i\s+want\s+to\s+block\s+my\s+card$", "म मेरो कार्ड ब्लक गर्न चाहन्छु।"),
    (r"^i\s+lost\s+my\s+(?:atm\s+)?card$", "मेरो कार्ड हराएको छ।"),
    (r"^(?:i\s+)?forgot\s+my\s+password(?:\s+please\s+reset\s+my\s+pin)?$", "म मेरो पासवर्ड बिर्सिएँ।"),
    (r"^(?:i\s+)?forgot\s+my\s+pin$", "म मेरो पिन बिर्सिएँ।"),
    (r"^please\s+reset\s+my\s+password$", "कृपया मेरो पासवर्ड रिसेट गर्नुहोस्।"),
    (r"^please\s+reset\s+my\s+pin$", "कृपया मेरो पिन रिसेट गर्नुहोस्।"),
    (r"^my\s+otp\s+is\s+not\s+coming$", "मेरो ओटीपी आइरहेको छैन।"),
    (r"^i\s+did\s+not\s+receive\s+otp$", "मैले ओटीपी प्राप्त गरिनँ।"),
    (r"^resend\s+otp$", "ओटीपी पुनः पठाउनुहोस्।"),
    (r"^i\s+want\s+to\s+open\s+an?\s+savings\s+account$", "म सेभिङ्स अकाउन्ट खोल्न चाहन्छु।"),
    (r"^i\s+want\s+to\s+open\s+an?\s+current\s+account$", "म करेन्ट अकाउन्ट खोल्न चाहन्छु।"),
    (r"^i\s+want\s+to\s+close\s+my\s+account$", "म मेरो अकाउन्ट बन्द गर्न चाहन्छु।"),
    (r"^what\s+is\s+my\s+account\s+balance$", "मेरो अकाउन्ट ब्यालेन्स कति छ?"),
    (r"^what\s+is\s+my\s+balance$", "मेरो ब्यालेन्स कति छ?"),
    (r"^check\s+my\s+balance$", "मेरो ब्यालेन्स चेक गर्नुहोस्।"),
    (r"^i\s+want\s+to\s+check\s+my\s+balance$", "म मेरो ब्यालेन्स चेक गर्न चाहन्छु।"),
    (r"^i\s+want\s+to\s+transfer\s+money$", "म पैसा ट्रान्सफर गर्न चाहन्छु।"),
    (r"^how\s+to\s+transfer\s+money$", "पैसा कसरी ट्रान्सफर गर्ने?"),
    (r"^how\s+to\s+send\s+money$", "पैसा कसरी पठाउने?"),
    (r"^i\s+want\s+to\s+send\s+money$", "म पैसा पठाउन चाहन्छु।"),
    (r"^transfer\s+money\s+to\s+my\s+account(?:\s+and\s+send\s+the\s+receipt)?$", "मेरो अकाउन्टमा पैसा ट्रान्सफर गर्नुहोस् र रसिद पठाउनुहोस्।"),
    (r"^i\s+want\s+to\s+deposit\s+money$", "म पैसा डिपोजिट गर्न चाहन्छु।"),
    (r"^i\s+want\s+to\s+withdraw\s+money$", "म पैसा विथड्र गर्न चाहन्छु।"),
    (r"^i\s+want\s+to\s+apply\s+for\s+a\s+loan$", "म लोनको लागि अप्लाई गर्न चाहन्छु।"),
    (r"^i\s+want\s+to\s+apply\s+for\s+a\s+loan\s+and\s+check\s+my\s+statement$", "म लोनको लागि अप्लाई गर्न चाहन्छु र मेरो स्टेटमेन्ट चेक गर्न चाहन्छु।"),
    (r"^i\s+need\s+my\s+statement$", "मलाई मेरो स्टेटमेन्ट चाहिन्छ।"),
    (r"^i\s+want\s+to\s+check\s+my\s+statement$", "म मेरो स्टेटमेन्ट हेर्न चाहन्छु।"),
    (r"^i\s+need\s+help\s+with\s+my\s+bank\s+account\s+balance$", "मलाई मेरो बैंक अकाउन्ट ब्यालेन्सको लागि हेल्प चाहिन्छ।"),
    (r"^i\s+need\s+help\s+with\s+my\s+bank\s+account$", "मलाई मेरो बैंक अकाउन्टको लागि हेल्प चाहिन्छ।"),
    (r"^i\s+need\s+help\s+please$", "कृपया मलाई हेल्प चाहिन्छ।"),
    (r"^please\s+help\s+me$", "कृपया मलाई हेल्प गर्नुहोस्।"),
    (r"^i\s+need\s+help$", "मलाई हेल्प चाहिन्छ।"),
    (r"^this\s+is\s+an\s+emergency(?:\s+call\s+the\s+ambulance)?$", "यो इमर्जेन्सी हो।"),
    (r"^emergency$", "इमर्जेन्सी।"),
]


def apply_strict_phrase_rules(text):
    if not text:
        return ""
    normalized = _normalize_spaces(text.lower())
    normalized = re.sub(r"[^\w\s]", "", normalized)
    for pattern, replacement in STRICT_PHRASE_RULES:
        if re.fullmatch(pattern, normalized):
            logger.info(f"   ✅ Strict rule HIT: {pattern}")
            return replacement
    return ""


# ══════════════════════════════════════════════════════════════
# SENTENCE DICTIONARY
# ══════════════════════════════════════════════════════════════
SENTENCE_DICT = {
    "my atm card is not working": "मेरो एटीएम कार्ड काम गरिरहेको छैन।",
    "my atm card is not working please help": "मेरो एटीएम कार्ड काम गरिरहेको छैन, कृपया हेल्प गर्नुहोस्।",
    "my card is not working": "मेरो कार्ड काम गरिरहेको छैन।",
    "my debit card is not working": "मेरो डेबिट कार्ड काम गरिरहेको छैन।",
    "my credit card is not working": "मेरो क्रेडिट कार्ड काम गरिरहेको छैन।",
    "my card is blocked": "मेरो कार्ड ब्लक भएको छ।",
    "please block my card": "कृपया मेरो कार्ड ब्लक गर्नुहोस्।",
    "i want to block my card": "म मेरो कार्ड ब्लक गर्न चाहन्छु।",
    "i lost my atm card": "मेरो एटीएम कार्ड हराएको छ।",
    "i lost my card": "मेरो कार्ड हराएको छ।",
    "atm is not working": "एटीएम काम गरिरहेको छैन।",
    "atm machine is not working": "एटीएम मेसिन काम गरिरहेको छैन।",
    "atm swallowed my card": "एटीएमले मेरो कार्ड निल्यो।",
    "i forgot my password": "म मेरो पासवर्ड बिर्सिएँ।",
    "i forgot my pin": "म मेरो पिन बिर्सिएँ।",
    "please reset my password": "कृपया मेरो पासवर्ड रिसेट गर्नुहोस्।",
    "please reset my pin": "कृपया मेरो पिन रिसेट गर्नुहोस्।",
    "my account is locked": "मेरो अकाउन्ट लक भएको छ।",
    "my otp is not coming": "मेरो ओटीपी आइरहेको छैन।",
    "i did not receive otp": "मैले ओटीपी प्राप्त गरिनँ।",
    "resend otp": "ओटीपी पुनः पठाउनुहोस्।",
    "i want to open an account": "म अकाउन्ट खोल्न चाहन्छु।",
    "i want to open a savings account": "म सेभिङ्स अकाउन्ट खोल्न चाहन्छु।",
    "i want to open a current account": "म करेन्ट अकाउन्ट खोल्न चाहन्छु।",
    "i want to close my account": "म मेरो अकाउन्ट बन्द गर्न चाहन्छु।",
    "what is my account balance": "मेरो अकाउन्ट ब्यालेन्स कति छ?",
    "what is my balance": "मेरो ब्यालेन्स कति छ?",
    "check my balance": "मेरो ब्यालेन्स चेक गर्नुहोस्।",
    "i want to check my balance": "म मेरो ब्यालेन्स चेक गर्न चाहन्छु।",
    "i want to transfer money": "म पैसा ट्रान्सफर गर्न चाहन्छु।",
    "transfer money to my account and send the receipt": "मेरो अकाउन्टमा पैसा ट्रान्सफर गर्नुहोस् र रसिद पठाउनुहोस्।",
    "how to transfer money": "पैसा कसरी ट्रान्सफर गर्ने?",
    "how to send money": "पैसा कसरी पठाउने?",
    "i want to send money": "म पैसा पठाउन चाहन्छु।",
    "i want to deposit money": "म पैसा डिपोजिट गर्न चाहन्छु।",
    "i want to withdraw money": "म पैसा विथड्र गर्न चाहन्छु।",
    "i want to apply for a loan": "म लोनको लागि अप्लाई गर्न चाहन्छु।",
    "i want to apply for a loan and check my statement": "म लोनको लागि अप्लाई गर्न चाहन्छु र मेरो स्टेटमेन्ट चेक गर्न चाहन्छु।",
    "i need my statement": "मलाई मेरो स्टेटमेन्ट चाहिन्छ।",
    "i want to check my statement": "म मेरो स्टेटमेन्ट हेर्न चाहन्छु।",
    "i need help": "मलाई हेल्प चाहिन्छ।",
    "please help me": "कृपया मलाई हेल्प गर्नुहोस्।",
    "i need help please": "कृपया मलाई हेल्प चाहिन्छ।",
    "i have a problem": "मलाई समस्या छ।",
    "there is a problem with my account": "मेरो अकाउन्टमा समस्या छ।",
    "what is your customer care number": "तपाईंको कस्टमर केयर नम्बर के हो?",
    "where is the nearest branch": "नजिकको ब्रान्च कहाँ छ?",
    "what are your working hours": "तपाईंको काम गर्ने समय के हो?",
    "this is an emergency": "यो इमर्जेन्सी हो।",
    "this is an emergency call the ambulance": "यो इमर्जेन्सी हो, एम्बुलेन्स कल गर्नुहोस्।",
    "emergency": "इमर्जेन्सी।",
}


# ══════════════════════════════════════════════════════════════
# LOANWORD DICTIONARY
# ══════════════════════════════════════════════════════════════
LOANWORD_DICT = {
    "bank": "बैंक",
    "account": "अकाउन्ट",
    "balance": "ब्यालेन्स",
    "loan": "लोन",
    "interest": "इन्टरेस्ट",
    "deposit": "डिपोजिट",
    "withdraw": "विथड्र",
    "withdrawal": "विथड्रल",
    "transfer": "ट्रान्सफर",
    "transaction": "ट्रान्जेक्शन",
    "statement": "स्टेटमेन्ट",
    "passbook": "पासबुक",
    "card": "कार्ड",
    "atm": "एटीएम",
    "debit": "डेबिट",
    "credit": "क्रेडिट",
    "pin": "पिन",
    "otp": "ओटीपी",
    "password": "पासवर्ड",
    "help": "हेल्प",
    "block": "ब्लक",
    "reset": "रिसेट",
    "receipt": "रसिद",
    "customer care": "कस्टमर केयर",
    "branch": "ब्रान्च",
    "mobile banking": "मोबाइल बैंकिङ",
    "internet banking": "इन्टरनेट बैंकिङ",
    "kyc": "केवाईसी",
    "complaint": "कम्प्लेन्ट",
    "refund": "रिफन्ड",
    "charge": "चार्ज",
    "emergency": "इमर्जेन्सी",
    "ambulance": "एम्बुलेन्स",
    "savings account": "सेभिङ्स अकाउन्ट",
    "current account": "करेन्ट अकाउन्ट",
}


WRONG_TRANSLATIONS = {
    "आपातकालीन": "इमर्जेन्सी",
    "आपातकाल": "इमर्जेन्सी",
    "आपत्काल": "इमर्जेन्सी",
    "आपत्कालीन": "इमर्जेन्सी",
    "आकस्मिक": "इमर्जेन्सी",
    "शेष": "ब्यालेन्स",
    "बाँकी रकम": "ब्यालेन्स",
    "अवशेष": "ब्यालेन्स",
    "कथन": "स्टेटमेन्ट",
    "विवरण": "स्टेटमेन्ट",
    "स्थानान्तरण": "ट्रान्सफर",
    "हस्तान्तरण": "ट्रान्सफर",
    "जम्मा": "डिपोजिट",
    "निकासी": "विथड्रल",
    "ऋण": "लोन",
    "कर्जा": "लोन",
    "व्याज": "इन्टरेस्ट",
    "ब्याज": "इन्टरेस्ट",
    "भुक्तानी": "पेमेन्ट",
    "खाता": "अकाउन्ट",
    "बचत": "सेभिङ्स",
    "मद्दत": "हेल्प",
    "सहायता": "हेल्प",
    "सहयोग": "हेल्प",
    "शाखा": "ब्रान्च",
    "अधिकारी": "अफिसर",
    "अस्पताल": "हस्पिटल",
    "बीमा": "इन्स्योरेन्स",
    "बिमा": "इन्स्योरेन्स",
    "गुनासो": "कम्प्लेन्ट",
    "उजुरी": "कम्प्लेन्ट",
    "मुद्दती निक्षेप": "फिक्स्ड डिपोजिट",
    "स्थायी निक्षेप": "फिक्स्ड डिपोजिट",
    "फिर्ता": "रिफन्ड",
    "कारोबार": "ट्रान्जेक्शन",
    "लेनदेन": "ट्रान्जेक्शन",
    "सीमा": "लिमिट",
    "निष्क्रिय": "इन्एक्टिभ",
    "अवरुद्ध": "ब्लक",
    "बन्द": "ब्लक",
}


def lookup_sentence_dict(text):
    if not text:
        return ""
    normalized = re.sub(r"[^\w\s]", "", text.lower()).strip()
    normalized = _normalize_spaces(normalized)
    if normalized in SENTENCE_DICT:
        logger.info(f"   ✅ Sentence dict HIT: {normalized}")
        return SENTENCE_DICT[normalized]
    for key, value in sorted(SENTENCE_DICT.items(), key=lambda x: len(x[0]), reverse=True):
        if key in normalized and len(key) > 10:
            logger.info(f"   ✅ Sentence dict PARTIAL: {key}")
            return value
    return ""


def apply_loanword_dict(text):
    if not text:
        return text
    result = text
    for english, nepali in sorted(LOANWORD_DICT.items(), key=lambda x: len(x[0]), reverse=True):
        pattern = r"\b" + re.escape(english) + r"\b"
        result = re.sub(pattern, nepali, result, flags=re.IGNORECASE)
    return result


def fix_wrong_translations(text):
    if not text:
        return text
    result = text
    for wrong, correct in sorted(WRONG_TRANSLATIONS.items(), key=lambda x: len(x[0]), reverse=True):
        result = result.replace(wrong, correct)
    return result


# ──────────────────────────────────────────────────────────────
# Audio Preprocessing
# ──────────────────────────────────────────────────────────────
def preprocess_audio(audio_bytes):
    with tempfile.NamedTemporaryFile(suffix=".webm", delete=False, dir="/tmp") as f:
        f.write(audio_bytes)
        raw_path = f.name

    wav_path = raw_path.replace(".webm", ".wav")
    exit_code = os.system(
        f"ffmpeg -y -i {raw_path} -ar 16000 -ac 1 "
        f"-c:a pcm_s16le {wav_path} -loglevel error"
    )
    return wav_path if exit_code == 0 else raw_path


# ──────────────────────────────────────────────────────────────
# Whisper Transcription
# ──────────────────────────────────────────────────────────────
def transcribe_audio(wav_path):
    logger.info("🎤 Transcribing...")
    result = whisper_model.transcribe(
        wav_path, language=None, task="transcribe",
        fp16=(DEVICE == "cuda"), best_of=5, beam_size=5,
        temperature=0.0, condition_on_previous_text=True,
        word_timestamps=False,
        initial_prompt=(
            "Bilingual Nepali-English. Preserve Devanagari for Nepali, "
            "Latin for English."
        ),
        suppress_tokens=[], without_timestamps=True,
    )
    return {
        "text": result["text"].strip(),
        "language": result.get("language", "ne"),
        "segments": result.get("segments", []),
    }


# ──────────────────────────────────────────────────────────────
# Translation
# ──────────────────────────────────────────────────────────────
def _normalize_bengali_to_devanagari(text):
    if transliterate and sanscript and _contains_bengali(text) and not _contains_devanagari(text):
        try:
            text = transliterate(text, sanscript.BENGALI, sanscript.DEVANAGARI)
        except Exception:
            pass
    return text


def _translate_to_nepali(text, retries=3):
    if not text or not text.strip() or GoogleTranslator is None:
        return ""

    for attempt in range(retries):
        try:
            translated = GoogleTranslator(source="auto", target="ne").translate(text[:4999])
            if not translated:
                time.sleep(1)
                continue

            translated = _normalize_bengali_to_devanagari(translated)
            if _is_valid_nepali_output(translated):
                return _sanitize_punctuation_nepali(translated)

            time.sleep(1)
        except Exception as e:
            logger.error(f"   Translation attempt {attempt + 1}: {e}")
            time.sleep(2)

    return ""


def _translate_segment_by_segment(text):
    if GoogleTranslator is None:
        return ""
    segments = re.split(r"(?<=[.!?।])\s+", text.strip())
    results = []

    for seg in segments:
        seg = seg.strip()
        if not seg:
            continue
        if _is_valid_nepali_output(seg):
            results.append(seg)
            continue
        t = _translate_to_nepali(seg)
        if t:
            results.append(t)

    return _sanitize_punctuation_nepali(" ".join(results)) if results else ""


# ──────────────────────────────────────────────────────────────
# Stage 2: Strict purification pipeline
# ──────────────────────────────────────────────────────────────
def purify_to_formal_nepali(raw_text):
    logger.info("🧹 Stage 2: Purifying...")

    if not raw_text or not raw_text.strip():
        return "कुनै पाठ प्राप्त भएन।"

    source = _normalize_spaces(raw_text)
    source_norm = re.sub(r"[^\w\s]", "", source.lower()).strip()
    logger.info(f"   Input: {repr(source)}")

    strict_hit = apply_strict_phrase_rules(source)
    if strict_hit:
        return _sanitize_punctuation_nepali(strict_hit)

    sentence_result = lookup_sentence_dict(source)
    if sentence_result:
        sentence_result = fix_wrong_translations(sentence_result)
        logger.info(f"   Sentence dict result: {sentence_result}")
        return _sanitize_punctuation_nepali(sentence_result)

    pre = apply_loanword_dict(source)
    logger.info(f"   After loanword pass: {pre}")

    if _is_valid_nepali_output(pre):
        result = fix_wrong_translations(pre)
        return _sanitize_punctuation_nepali(result)

    if len(source_norm.split()) <= 14:
        candidate = _translate_to_nepali(pre)
    else:
        candidate = _translate_segment_by_segment(pre)

    if not _is_valid_nepali_output(candidate):
        logger.info("   Final retry on original source...")
        candidate = _translate_to_nepali(source)

    if not _is_valid_nepali_output(candidate):
        logger.error("❌ All attempts failed.")
        return "अनुवाद गर्न सकिएन। कृपया पुनः प्रयास गर्नुहोस्।"

    candidate = apply_loanword_dict(candidate)
    candidate = fix_wrong_translations(candidate)
    if not _is_valid_nepali_output(candidate):
        logger.error("❌ Output too weak after cleanup.")
        return "अनुवाद गर्न सकिएन। कृपया पुनः प्रयास गर्नुहोस्।"

    return _sanitize_punctuation_nepali(candidate)


# ──────────────────────────────────────────────────────────────
# Stage 3: Romanization
# ──────────────────────────────────────────────────────────────
DEVANAGARI_TO_ROMAN = {
    "अ":"a","आ":"aa","इ":"i","ई":"ii","उ":"u","ऊ":"uu",
    "ए":"e","ऐ":"ai","ओ":"o","औ":"au","अं":"am","अः":"ah",
    "क":"ka","ख":"kha","ग":"ga","घ":"gha","ङ":"nga",
    "च":"cha","छ":"chha","ज":"ja","झ":"jha","ञ":"nya",
    "ट":"ta","ठ":"tha","ड":"da","ढ":"dha","ण":"na",
    "त":"ta","थ":"tha","द":"da","ध":"dha","न":"na",
    "प":"pa","फ":"pha","ब":"ba","भ":"bha","म":"ma",
    "य":"ya","र":"ra","ल":"la","व":"wa","श":"sha",
    "ष":"sha","स":"sa","ह":"ha","क्ष":"ksha","त्र":"tra",
    "ज्ञ":"gya","ा":"aa","ि":"i","ी":"ii","ु":"u",
    "ू":"uu","े":"e","ै":"ai","ो":"o","ौ":"au",
    "ं":"m","ः":"h","्":"","।":".","॥":"..","ँ":"n",
    "०":"0","१":"1","२":"2","३":"3","४":"4",
    "५":"5","६":"6","७":"7","८":"8","९":"9",
}


def fallback_romanize(text):
    result, i = [], 0
    while i < len(text):
        two = text[i:i + 2]
        if two in DEVANAGARI_TO_ROMAN:
            result.append(DEVANAGARI_TO_ROMAN[two]); i += 2
        elif text[i] in DEVANAGARI_TO_ROMAN:
            result.append(DEVANAGARI_TO_ROMAN[text[i]]); i += 1
        else:
            result.append(text[i]); i += 1
    return "".join(result)


def post_process_romanization(text):
    for iast, roman in {
        "ā":"aa","ī":"ii","ū":"uu","ṛ":"ri","ṭ":"t","ḍ":"d",
        "ṇ":"n","ś":"sh","ṣ":"sh","ñ":"n","ṃ":"m","ḥ":"h",
        "ṅ":"ng","।":".","॥":"..",
    }.items():
        text = text.replace(iast, roman)
    return text


def romanize_nepali(text):
    if not text or not _contains_devanagari(text):
        return ""
    if transliterate and sanscript:
        try:
            r = transliterate(text, sanscript.DEVANAGARI, sanscript.IAST)
            return post_process_romanization(r)
        except Exception:
            pass
    return fallback_romanize(text)


# ──────────────────────────────────────────────────────────────
# Self-test
# ──────────────────────────────────────────────────────────────
print("🧪 Self-test...")
print(f"   Strict rules:    {len(STRICT_PHRASE_RULES)} entries")
print(f"   Sentence dict:   {len(SENTENCE_DICT)} entries")
print(f"   Loanword dict:   {len(LOANWORD_DICT)} words")
print(f"   Wrong-fix table: {len(WRONG_TRANSLATIONS)} entries")

_tests = [
    "My ATM card is not working, please help.",
    "I forgot my password, please reset my PIN.",
    "This is an emergency, call the ambulance.",
    "I want to apply for a loan and check my statement.",
    "I need help with my bank account balance.",
    "Transfer money to my account and send the receipt.",
    "What is my account balance?",
    "I want to open a savings account.",
    "Please block my card.",
]

for t in _tests:
    out = purify_to_formal_nepali(t)
    rom = romanize_nepali(out)
    ok = "✅" if _is_valid_nepali_output(out) else "❌"
    src = "📖" if lookup_sentence_dict(t) or apply_strict_phrase_rules(t) else "🔄"
    print(f"\n{ok}{src} IN:  {t}")
    print(f"    NE:  {out}")
    print(f"    ROM: {rom}")

print(f"\n✅ Cell 6 done!")
print(f"   📖 = answered by strict rules or sentence dictionary")
print(f"   🔄 = translated only as a fallback")

🧪 Self-test...
   Strict rules:    38 entries
   Sentence dict:   51 entries
   Loanword dict:   35 words
   Wrong-fix table: 40 entries

✅📖 IN:  My ATM card is not working, please help.
    NE:  मेरो एटीएम कार्ड काम गरिरहेको छैन।
    ROM: maerao etaiiema kaaarada kaaama garairahaekao chhaaina.

✅📖 IN:  I forgot my password, please reset my PIN.
    NE:  म मेरो पासवर्ड बिर्सिएँ।
    ROM: ma maerao paaasawarada bairasaien.

✅📖 IN:  This is an emergency, call the ambulance.
    NE:  यो इमर्जेन्सी हो।
    ROM: yao imarajaenasaii hao.

✅📖 IN:  I want to apply for a loan and check my statement.
    NE:  म लोनको लागि अप्लाई गर्न चाहन्छु र मेरो स्टेटमेन्ट चेक गर्न चाहन्छु।
    ROM: ma laonakao laaagai apalaaaii garana chaaahanachhau ra maerao sataetamaenata chaeka garana chaaahanachhau.

✅📖 IN:  I need help with my bank account balance.
    NE:  मलाई मेरो बैंक अकाउन्ट ब्यालेन्सको लागि हेल्प चाहिन्छ।
    ROM: malaaaii maerao baaimka akaaaunata bayaaalaenasakao laaagai haelapa chaaahainachha.



In [7]:
# ============================================================
# CELL 7: Flask App
# ============================================================
import traceback, base64

app = Flask(__name__)
CORS(app, origins="*")

@app.route("/", methods=["GET"])
def serve_index():
    return send_from_directory("/content", "index.html")

@app.route("/health", methods=["GET"])
def health_check():
    return jsonify({
        "status": "online", "device": DEVICE,
        "models": {
            "whisper": "large-v3",
            "translator": "deep-translator",
            "romanizer": "indic_transliteration",
            "loanwords": f"{len(LOANWORD_DICT)} words",
            "wrong_fix": f"{len(WRONG_TRANSLATIONS)} fixes"
        },
        "gpu_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    })

@app.route("/process-audio", methods=["POST"])
def process_audio():
    logger.info("=" * 50)

    if "audio" not in request.files:
        return jsonify({"error": "No audio file"}), 400

    try:
        audio_bytes = request.files["audio"].read()
        if len(audio_bytes) < 1000:
            return jsonify({"error": "Audio too short"}), 400
    except Exception as e:
        return jsonify({"error": str(e)}), 500

    # Store audio bytes for playback (base64 encode)
    audio_b64 = base64.b64encode(audio_bytes).decode("utf-8")
    audio_mime = request.files["audio"].content_type or "audio/webm"

    try:
        wav_path = preprocess_audio(audio_bytes)
    except Exception as e:
        return jsonify({"error": f"Preprocessing failed: {e}"}), 500

    try:
        tr = transcribe_audio(wav_path)
        raw_spoken = tr["text"]
        detected_language = tr["language"]
        if not raw_spoken:
            return jsonify({"error": "No speech detected"}), 422
    except Exception as e:
        return jsonify({"error": f"Transcription failed: {e}"}), 500

    try:
        pure_nepali = purify_to_formal_nepali(raw_spoken)
    except Exception as e:
        logger.error(traceback.format_exc())
        pure_nepali = "अनुवाद गर्न सकिएन।"

    try:
        romanized_nepali = romanize_nepali(pure_nepali)
    except Exception:
        romanized_nepali = ""

    # Cleanup
    try:
        os.remove(wav_path)
        webm = wav_path.replace(".wav", ".webm")
        if os.path.exists(webm): os.remove(webm)
    except Exception:
        pass

    return jsonify({
        "raw_spoken":       raw_spoken,
        "pure_nepali":      pure_nepali,
        "romanized_nepali": romanized_nepali,
        "audio_b64":        audio_b64,
        "audio_mime":       audio_mime,
        "metadata": {
            "detected_language":   detected_language,
            "audio_bytes":         len(audio_bytes),
            "processing_device":   DEVICE,
            "loanwords_dictionary": f"{len(LOANWORD_DICT)} words",
        }
    }), 200

@app.errorhandler(413)
def too_large(e): return jsonify({"error": "File too large"}), 413

@app.errorhandler(500)
def server_error(e): return jsonify({"error": str(e)}), 500

print("✅ Flask app ready!")

✅ Flask app ready!


In [8]:
# ============================================================
# CELL 8: Frontend HTML with Playback Feature
# ============================================================

HTML_CONTENT = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Nepali Speech Processor</title>
  <style>
    *,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
    :root{
      --primary:#6C63FF;--primary-dark:#5A52D5;
      --danger:#FF4757;--danger-dark:#E84057;
      --success:#2ED573;--warning:#FFA502;
      --bg-dark:#0D0D1A;--bg-card:#16162A;
      --bg-input:#1E1E35;--border:#2A2A4A;
      --text-primary:#EAEAF5;--text-secondary:#8888AA;
      --text-accent:#A89CFF;
      --glow-primary:rgba(108,99,255,0.4);
      --glow-danger:rgba(255,71,87,0.4);
    }
    body{
      font-family:'Segoe UI',system-ui,sans-serif;
      background:var(--bg-dark);color:var(--text-primary);
      min-height:100vh;overflow-x:hidden;
    }
    body::before{
      content:'';position:fixed;inset:0;
      background:
        radial-gradient(ellipse 80% 50% at 20% 20%,rgba(108,99,255,0.08) 0%,transparent 60%),
        radial-gradient(ellipse 60% 40% at 80% 80%,rgba(255,71,87,0.05) 0%,transparent 60%);
      pointer-events:none;z-index:0;
    }
    .container{
      position:relative;z-index:1;
      max-width:900px;margin:0 auto;
      padding:2rem 1.5rem 4rem;
    }
    header{text-align:center;margin-bottom:2.5rem}
    .logo{font-size:2.8rem;display:block;filter:drop-shadow(0 0 20px var(--glow-primary));margin-bottom:.5rem}
    h1{
      font-size:clamp(1.4rem,4vw,2.1rem);font-weight:700;
      background:linear-gradient(135deg,#EAEAF5 0%,#A89CFF 50%,#6C63FF 100%);
      -webkit-background-clip:text;-webkit-text-fill-color:transparent;
      background-clip:text;line-height:1.3;margin-bottom:.5rem;
    }
    .subtitle{color:var(--text-secondary);font-size:.9rem}
    .badge-row{display:flex;gap:.5rem;justify-content:center;flex-wrap:wrap;margin-top:.75rem}
    .badge{
      padding:.25rem .8rem;border-radius:100px;font-size:.7rem;font-weight:600;
      background:rgba(108,99,255,0.12);border:1px solid rgba(108,99,255,0.25);
      color:var(--text-accent);
    }

    /* ── Control Panel ─────────────────────────────────── */
    .control-panel{
      background:var(--bg-card);border:1px solid var(--border);
      border-radius:20px;padding:2rem;margin-bottom:1.5rem;
      text-align:center;box-shadow:0 8px 32px rgba(0,0,0,.3);
    }
    .visualizer-wrap{position:relative;margin-bottom:1.5rem}
    #waveformCanvas{
      width:100%;height:80px;border-radius:12px;
      background:var(--bg-input);border:1px solid var(--border);display:block;
    }
    .viz-label{
      position:absolute;top:50%;left:50%;transform:translate(-50%,-50%);
      color:var(--text-secondary);font-size:.78rem;pointer-events:none;transition:opacity .3s;
    }
    .viz-label.hidden{opacity:0}

    /* ── Record Button ─────────────────────────────────── */
    .record-wrap{display:inline-flex;flex-direction:column;align-items:center;gap:.75rem;margin-bottom:1.25rem}
    #recordBtn{
      width:96px;height:96px;border-radius:50%;border:none;cursor:pointer;
      background:linear-gradient(135deg,var(--primary),var(--primary-dark));
      box-shadow:0 0 0 0 var(--glow-primary),0 8px 24px rgba(0,0,0,.4);
      transition:transform .2s;outline:none;
    }
    #recordBtn:hover{transform:scale(1.06)}
    #recordBtn:active{transform:scale(.96)}
    .btn-icon{font-size:2.1rem;line-height:1;display:block}
    #recordBtn.recording{
      background:linear-gradient(135deg,var(--danger),var(--danger-dark));
      animation:pulse-ring 1.5s ease infinite;
    }
    @keyframes pulse-ring{
      0%  {box-shadow:0 0 0 0 var(--glow-danger),0 8px 24px rgba(0,0,0,.4)}
      70% {box-shadow:0 0 0 18px rgba(255,71,87,0),0 8px 24px rgba(0,0,0,.4)}
      100%{box-shadow:0 0 0 0 rgba(255,71,87,0),0 8px 24px rgba(0,0,0,.4)}
    }
    .btn-label{font-size:.85rem;font-weight:600;color:var(--text-secondary);text-transform:uppercase;letter-spacing:.5px}
    .timer{font-size:1.9rem;font-weight:700;color:var(--danger);letter-spacing:2px;display:none;min-height:2.4rem}
    .timer.visible{display:block}

    /* ── Status ────────────────────────────────────────── */
    .status-badge{
      display:inline-flex;align-items:center;gap:.5rem;
      padding:.45rem 1.1rem;border-radius:100px;font-size:.82rem;font-weight:600;
      background:var(--bg-input);border:1px solid var(--border);
      color:var(--text-secondary);transition:all .3s;
    }
    .status-dot{width:8px;height:8px;border-radius:50%;background:var(--text-secondary);transition:all .3s}
    .status-badge.ready .status-dot{background:var(--success);box-shadow:0 0 6px var(--success)}
    .status-badge.recording .status-dot{background:var(--danger);box-shadow:0 0 6px var(--danger);animation:blink 1s step-end infinite}
    .status-badge.processing .status-dot{background:var(--warning);box-shadow:0 0 6px var(--warning);animation:spin-dot 1s linear infinite}
    .status-badge.success .status-dot{background:var(--success);box-shadow:0 0 8px var(--success)}
    .status-badge.error .status-dot{background:var(--danger);box-shadow:0 0 8px var(--danger)}
    @keyframes blink{0%,100%{opacity:1}50%{opacity:0}}
    @keyframes spin-dot{0%{transform:scale(1);opacity:1}50%{transform:scale(1.5);opacity:.7}100%{transform:scale(1);opacity:1}}

    /* ── Progress ──────────────────────────────────────── */
    .progress-wrap{margin-top:1.25rem;display:none}
    .progress-wrap.visible{display:block}
    .progress-stages{display:flex;justify-content:space-around;margin-bottom:.5rem}
    .stage-item{display:flex;flex-direction:column;align-items:center;gap:.2rem;flex:1}
    .stage-icon{font-size:1rem;opacity:.35;transition:opacity .3s,transform .3s}
    .stage-item.active .stage-icon{opacity:1;transform:scale(1.2)}
    .stage-item.done .stage-icon{opacity:1;color:var(--success)}
    .stage-label{font-size:.6rem;color:var(--text-secondary);text-align:center}
    .progress-track{height:4px;background:var(--bg-input);border-radius:2px;overflow:hidden}
    .progress-fill{height:100%;width:0%;background:linear-gradient(90deg,var(--primary),var(--success));border-radius:2px;transition:width .5s ease}

    /* ── Playback Panel ────────────────────────────────── */
    .playback-panel{
      background:var(--bg-card);border:1px solid var(--border);
      border-radius:16px;padding:1.25rem 1.5rem;
      margin-bottom:1.5rem;display:none;
    }
    .playback-panel.visible{display:block}
    .playback-header{
      display:flex;align-items:center;gap:.6rem;
      margin-bottom:1rem;
    }
    .playback-title{font-size:.78rem;font-weight:700;text-transform:uppercase;letter-spacing:.8px;color:var(--text-secondary)}
    .playback-controls{display:flex;align-items:center;gap:1rem;flex-wrap:wrap}
    .play-btn{
      display:flex;align-items:center;gap:.5rem;
      padding:.5rem 1.2rem;border-radius:10px;border:none;cursor:pointer;
      background:linear-gradient(135deg,var(--primary),var(--primary-dark));
      color:#fff;font-size:.85rem;font-weight:600;transition:transform .2s,opacity .2s;
    }
    .play-btn:hover{transform:scale(1.04);opacity:.9}
    .play-btn:active{transform:scale(.97)}
    .play-btn:disabled{opacity:.4;cursor:not-allowed;transform:none}
    .audio-progress-wrap{flex:1;min-width:160px}
    .audio-track{
      width:100%;height:5px;border-radius:3px;
      -webkit-appearance:none;appearance:none;
      background:var(--bg-input);outline:none;cursor:pointer;
    }
    .audio-track::-webkit-slider-thumb{
      -webkit-appearance:none;width:14px;height:14px;
      border-radius:50%;background:var(--primary);cursor:pointer;
    }
    .audio-time{font-size:.75rem;color:var(--text-secondary);white-space:nowrap;font-variant-numeric:tabular-nums}
    .volume-wrap{display:flex;align-items:center;gap:.4rem}
    .vol-icon{font-size:.9rem}
    .vol-slider{
      width:70px;height:4px;border-radius:2px;
      -webkit-appearance:none;appearance:none;
      background:var(--bg-input);outline:none;cursor:pointer;
    }
    .vol-slider::-webkit-slider-thumb{
      -webkit-appearance:none;width:12px;height:12px;
      border-radius:50%;background:var(--text-accent);cursor:pointer;
    }

    /* ── Result Cards ──────────────────────────────────── */
    .results-grid{display:grid;gap:1.25rem}
    .result-card{
      background:var(--bg-card);border:1px solid var(--border);
      border-radius:16px;padding:1.4rem;
      box-shadow:0 4px 16px rgba(0,0,0,.2);
      transition:border-color .3s,box-shadow .3s;position:relative;overflow:hidden;
    }
    .result-card::before{content:'';position:absolute;top:0;left:0;right:0;height:3px;border-radius:16px 16px 0 0}
    .card-raw::before    {background:linear-gradient(90deg,#FF6B6B,#FF8E53)}
    .card-nepali::before {background:linear-gradient(90deg,#6C63FF,#A89CFF)}
    .card-roman::before  {background:linear-gradient(90deg,#2ED573,#1DD1A1)}
    .result-card:hover{border-color:rgba(108,99,255,.4);box-shadow:0 8px 24px rgba(108,99,255,.1)}
    .card-header{display:flex;align-items:center;justify-content:space-between;margin-bottom:.9rem}
    .card-title{display:flex;align-items:center;gap:.5rem}
    .card-icon{font-size:1.1rem}
    .card-label{font-size:.72rem;font-weight:700;text-transform:uppercase;letter-spacing:.8px;color:var(--text-secondary)}
    .card-badge{font-size:.62rem;font-weight:600;padding:.15rem .55rem;border-radius:100px;border:1px solid var(--border);color:var(--text-secondary)}
    .copy-btn{
      background:none;border:1px solid var(--border);border-radius:8px;
      color:var(--text-secondary);cursor:pointer;padding:.28rem .55rem;
      font-size:.72rem;transition:all .2s;display:flex;align-items:center;gap:.3rem;
    }
    .copy-btn:hover{background:var(--bg-input);color:var(--text-primary);border-color:var(--primary)}
    .result-text{font-size:1.05rem;line-height:1.7;color:var(--text-primary);min-height:2.2rem;word-break:break-word}
    .result-text.nepali-font{font-size:1.2rem;font-family:'Noto Sans Devanagari','Mangal',serif}
    .result-text.placeholder{color:var(--text-secondary);font-style:italic;font-size:.88rem}

    /* ── Meta Strip ────────────────────────────────────── */
    .meta-strip{margin-top:1rem;padding-top:1rem;border-top:1px solid var(--border);display:flex;gap:1.5rem;flex-wrap:wrap}
    .meta-item{font-size:.7rem;color:var(--text-secondary)}
    .meta-item strong{color:var(--text-accent)}

    /* ── Toast ─────────────────────────────────────────── */
    .toast-container{position:fixed;bottom:2rem;right:1.5rem;z-index:9999;display:flex;flex-direction:column;gap:.5rem}
    .toast{
      background:#2A1520;border:1px solid var(--danger);border-radius:12px;
      padding:.9rem 1.1rem;max-width:320px;color:var(--text-primary);
      font-size:.85rem;box-shadow:0 8px 24px rgba(0,0,0,.4);animation:slide-in .3s ease;
    }
    .toast.success-toast{background:#0D2A1A;border-color:var(--success)}
    @keyframes slide-in{from{transform:translateX(120%);opacity:0}to{transform:translateX(0);opacity:1}}

    @media(max-width:600px){
      .control-panel{padding:1.25rem .9rem}
      #recordBtn{width:80px;height:80px}
      .btn-icon{font-size:1.8rem}
      .playback-controls{flex-direction:column;align-items:flex-start}
    }
  </style>
  <link rel="preconnect" href="https://fonts.googleapis.com"/>
  <link href="https://fonts.googleapis.com/css2?family=Noto+Sans+Devanagari:wght@400;600&display=swap" rel="stylesheet"/>
</head>
<body>
<div class="container">

  <header>
    <span class="logo">🎙️</span>
    <h1>Nepali-English Speech Processor</h1>
    <p class="subtitle">Real-time ASR · Translation · Romanization</p>
    <div class="badge-row">
      <span class="badge">📚 500+ Word Dictionary</span>
      <span class="badge">🏦 Banking Phrases</span>
      <span class="badge">🔊 Audio Playback</span>
      <span class="badge">🇳🇵 Loanword Protected</span>
    </div>
  </header>

  <!-- Control Panel -->
  <div class="control-panel">
    <div class="visualizer-wrap">
      <canvas id="waveformCanvas" height="80"></canvas>
      <span class="viz-label" id="vizLabel">Microphone waveform appears here while recording</span>
    </div>

    <div class="record-wrap">
      <button id="recordBtn" aria-label="Start recording">
        <span class="btn-icon" id="btnIcon">🎤</span>
      </button>
      <span class="btn-label" id="btnLabel">Tap to Record</span>
    </div>

    <div class="timer" id="timer">00:00</div>

    <div class="status-badge idle" id="statusBadge">
      <div class="status-dot"></div>
      <span id="statusText">Ready — tap the mic to start</span>
    </div>

    <div class="progress-wrap" id="progressWrap">
      <div class="progress-stages">
        <div class="stage-item" id="stage-asr">
          <span class="stage-icon">🎤</span>
          <span class="stage-label">Transcribing</span>
        </div>
        <div class="stage-item" id="stage-nlp">
          <span class="stage-icon">🔄</span>
          <span class="stage-label">Translating</span>
        </div>
        <div class="stage-item" id="stage-roman">
          <span class="stage-icon">🔤</span>
          <span class="stage-label">Romanizing</span>
        </div>
        <div class="stage-item" id="stage-done">
          <span class="stage-icon">✅</span>
          <span class="stage-label">Done</span>
        </div>
      </div>
      <div class="progress-track">
        <div class="progress-fill" id="progressFill"></div>
      </div>
    </div>
  </div>

  <!-- Playback Panel -->
  <div class="playback-panel" id="playbackPanel">
    <div class="playback-header">
      <span style="font-size:1.1rem">🔊</span>
      <span class="playback-title">Recording Playback</span>
    </div>
    <div class="playback-controls">
      <button class="play-btn" id="playBtn" onclick="togglePlay()" disabled>
        <span id="playIcon">▶</span>
        <span id="playLabel">Play</span>
      </button>
      <div class="audio-progress-wrap">
        <input type="range" class="audio-track" id="audioTrack"
               min="0" max="100" value="0" oninput="seekAudio(this.value)"/>
      </div>
      <span class="audio-time" id="audioTime">0:00 / 0:00</span>
      <div class="volume-wrap">
        <span class="vol-icon">🔈</span>
        <input type="range" class="vol-slider" id="volSlider"
               min="0" max="1" step="0.05" value="1" oninput="setVolume(this.value)"/>
        <span class="vol-icon">🔊</span>
      </div>
    </div>
    <audio id="audioPlayer" preload="auto"></audio>
  </div>

  <!-- Results -->
  <div class="results-grid">
    <div class="result-card card-raw">
      <div class="card-header">
        <div class="card-title">
          <span class="card-icon">📝</span>
          <span class="card-label">Raw Transcription</span>
          <span class="card-badge">Stage 1</span>
        </div>
        <button class="copy-btn" onclick="copyText(\'rawText\')">📋 Copy</button>
      </div>
      <div class="result-text placeholder" id="rawText">
        Verbatim speech — Nepali &amp; English as spoken…
      </div>
    </div>

    <div class="result-card card-nepali">
      <div class="card-header">
        <div class="card-title">
          <span class="card-icon">🇳🇵</span>
          <span class="card-label">Pure Formal Nepali</span>
          <span class="card-badge">Stage 2</span>
        </div>
        <button class="copy-btn" onclick="copyText(\'nepaliText\')">📋 Copy</button>
      </div>
      <div class="result-text nepali-font placeholder" id="nepaliText">
        शुद्ध नेपाली देवनागरी…
      </div>
    </div>

    <div class="result-card card-roman">
      <div class="card-header">
        <div class="card-title">
          <span class="card-icon">🔤</span>
          <span class="card-label">Romanized Nepali</span>
          <span class="card-badge">Stage 3</span>
        </div>
        <button class="copy-btn" onclick="copyText(\'romanText\')">📋 Copy</button>
      </div>
      <div class="result-text placeholder" id="romanText">
        Phonetic romanization will appear here…
      </div>
    </div>
  </div>

  <div class="meta-strip" id="metaStrip" style="display:none">
    <div class="meta-item">Lang: <strong id="metaLang">—</strong></div>
    <div class="meta-item">Audio: <strong id="metaBytes">—</strong></div>
    <div class="meta-item">Device: <strong id="metaDevice">—</strong></div>
    <div class="meta-item">Time: <strong id="metaTime">—</strong></div>
    <div class="meta-item">Dict: <strong id="metaDict">—</strong></div>
  </div>

</div>
<div class="toast-container" id="toastContainer"></div>

<script>
const API_BASE_URL = "NGROK_URL_PLACEHOLDER";

// DOM
const recordBtn    = document.getElementById("recordBtn");
const btnIcon      = document.getElementById("btnIcon");
const btnLabel     = document.getElementById("btnLabel");
const statusBadge  = document.getElementById("statusBadge");
const statusText   = document.getElementById("statusText");
const timerEl      = document.getElementById("timer");
const progressWrap = document.getElementById("progressWrap");
const progressFill = document.getElementById("progressFill");
const canvas       = document.getElementById("waveformCanvas");
const vizLabel     = document.getElementById("vizLabel");
const ctx2d        = canvas.getContext("2d");
const rawTextEl    = document.getElementById("rawText");
const nepaliTextEl = document.getElementById("nepaliText");
const romanTextEl  = document.getElementById("romanText");
const metaStrip    = document.getElementById("metaStrip");

// Audio Player
const audioPlayer   = document.getElementById("audioPlayer");
const playBtn       = document.getElementById("playBtn");
const playIcon      = document.getElementById("playIcon");
const playLabel     = document.getElementById("playLabel");
const audioTrack    = document.getElementById("audioTrack");
const audioTime     = document.getElementById("audioTime");
const volSlider     = document.getElementById("volSlider");
const playbackPanel = document.getElementById("playbackPanel");

// State
let mediaRecorder = null, audioChunks = [], isRecording = false;
let timerInterval = null, timerSeconds = 0;
let animId = null, analyser = null, audioCtx = null, micStream = null;

// ── Resize Canvas ───────────────────────────────────────────
function resizeCanvas() {
  const r = canvas.getBoundingClientRect();
  canvas.width  = r.width  || 640;
  canvas.height = r.height || 80;
}
resizeCanvas();
window.addEventListener("resize", resizeCanvas);

// ── Waveform Visualizer ─────────────────────────────────────
function startVisualizer(stream) {
  audioCtx = new (window.AudioContext || window.webkitAudioContext)();
  analyser = audioCtx.createAnalyser();
  analyser.fftSize = 256;
  audioCtx.createMediaStreamSource(stream).connect(analyser);
  const data = new Uint8Array(analyser.frequencyBinCount);
  vizLabel.classList.add("hidden");

  function draw() {
    animId = requestAnimationFrame(draw);
    analyser.getByteTimeDomainData(data);
    const W = canvas.width, H = canvas.height;
    ctx2d.fillStyle = "#1E1E35";
    ctx2d.fillRect(0, 0, W, H);
    ctx2d.lineWidth = 2; ctx2d.strokeStyle = "#6C63FF";
    ctx2d.shadowBlur = 8; ctx2d.shadowColor = "rgba(108,99,255,.6)";
    ctx2d.beginPath();
    const slice = W / data.length;
    let x = 0;
    data.forEach((v, i) => {
      const y = (v / 128) * (H / 2);
      i === 0 ? ctx2d.moveTo(x, y) : ctx2d.lineTo(x, y);
      x += slice;
    });
    ctx2d.lineTo(W, H / 2); ctx2d.stroke(); ctx2d.shadowBlur = 0;
  }
  draw();
}

function stopVisualizer() {
  if (animId) { cancelAnimationFrame(animId); animId = null; }
  const W = canvas.width, H = canvas.height;
  ctx2d.clearRect(0, 0, W, H);
  ctx2d.strokeStyle = "rgba(108,99,255,.25)"; ctx2d.lineWidth = 1;
  ctx2d.beginPath();
  ctx2d.moveTo(0, H / 2); ctx2d.lineTo(W, H / 2); ctx2d.stroke();
  vizLabel.classList.remove("hidden");
  if (audioCtx) { audioCtx.close().catch(() => {}); audioCtx = null; }
}

// ── Timer ───────────────────────────────────────────────────
function startTimer() {
  timerSeconds = 0;
  timerEl.classList.add("visible");
  timerEl.textContent = "00:00";
  timerInterval = setInterval(() => {
    timerSeconds++;
    const m = String(Math.floor(timerSeconds / 60)).padStart(2, "0");
    const s = String(timerSeconds % 60).padStart(2, "0");
    timerEl.textContent = `${m}:${s}`;
    if (timerSeconds >= 120) {
      showToast("⏱️ Max recording time reached", "warning");
      toggleRecording();
    }
  }, 1000);
}

function stopTimer() {
  clearInterval(timerInterval);
  timerEl.classList.remove("visible");
}

// ── Status ──────────────────────────────────────────────────
function setStatus(state, msg) {
  statusBadge.className = `status-badge ${state}`;
  statusText.textContent = msg;
}

// ── Progress ────────────────────────────────────────────────
const stages = ["asr", "nlp", "roman", "done"];
function showProgress() {
  progressWrap.classList.add("visible");
  progressFill.style.width = "0%";
  stages.forEach(s => document.getElementById(`stage-${s}`).className = "stage-item");
}
function advanceProgress(idx) {
  stages.forEach((s, i) => {
    const el = document.getElementById(`stage-${s}`);
    el.className = i < idx ? "stage-item done" : i === idx ? "stage-item active" : "stage-item";
  });
  progressFill.style.width = `${((idx + 1) / stages.length) * 100}%`;
}
function hideProgress() {
  setTimeout(() => progressWrap.classList.remove("visible"), 2000);
}

// ── Recording ───────────────────────────────────────────────
async function toggleRecording() {
  isRecording ? stopRecording() : await startRecording();
}

async function startRecording() {
  try {
    micStream = await navigator.mediaDevices.getUserMedia({
      audio: {
        channelCount: 1, sampleRate: 16000,
        echoCancellation: true, noiseSuppression: true, autoGainControl: true
      }
    });
  } catch (err) {
    showToast(`❌ Mic denied: ${err.message}`, "error");
    setStatus("error", "Microphone permission required");
    return;
  }

  const mime = ["audio/webm;codecs=opus","audio/webm","audio/ogg;codecs=opus","audio/ogg","audio/mp4"]
    .find(t => MediaRecorder.isTypeSupported(t)) || "";

  mediaRecorder = new MediaRecorder(micStream, mime ? { mimeType: mime } : {});
  audioChunks = [];
  mediaRecorder.ondataavailable = e => { if (e.data.size > 0) audioChunks.push(e.data); };
  mediaRecorder.onstop = handleAudioReady;
  mediaRecorder.start(250);

  isRecording = true;
  startVisualizer(micStream);
  startTimer();

  recordBtn.classList.add("recording");
  btnIcon.textContent = "⏹️"; btnLabel.textContent = "Tap to Stop";
  setStatus("recording", "🔴 Recording — speak now…");

  clearResults();

  // Hide playback panel when starting new recording
  playbackPanel.classList.remove("visible");
}

function stopRecording() {
  if (mediaRecorder && mediaRecorder.state !== "inactive") mediaRecorder.stop();
  micStream?.getTracks().forEach(t => t.stop());
  isRecording = false;
  stopVisualizer(); stopTimer();
  recordBtn.classList.remove("recording");
  btnIcon.textContent = "🎤"; btnLabel.textContent = "Tap to Record";
  setStatus("processing", "⚙️ Processing…");
  showProgress(); advanceProgress(0);
}

// ── Send to Server ──────────────────────────────────────────
async function handleAudioReady() {
  if (!audioChunks.length) {
    setStatus("error", "No audio recorded");
    showToast("❌ No audio captured", "error");
    return;
  }

  const mimeType = mediaRecorder.mimeType || "audio/webm";
  const blob     = new Blob(audioChunks, { type: mimeType });

  // ── Setup local playback immediately ───────────────────────
  // User can listen to their recording while server processes
  const localURL = URL.createObjectURL(blob);
  setupAudioPlayer(localURL, mimeType);

  const fd = new FormData();
  fd.append("audio", blob, "recording.webm");
  const t0 = performance.now();

  try {
    advanceProgress(1);
    setStatus("processing", "🔄 Sending to server…");

    const res = await fetch(`${API_BASE_URL}/process-audio`, {
      method: "POST", body: fd, headers: { "Accept": "application/json" }
    });
    advanceProgress(2);

    if (!res.ok) {
      const err = await res.json().catch(() => ({}));
      throw new Error(err.error || `HTTP ${res.status}`);
    }

    const data  = await res.json();
    const elapsed = ((performance.now() - t0) / 1000).toFixed(1);
    advanceProgress(3);
    setStatus("success", "✅ Done!");
    displayResults(data, elapsed);
    showToast("✅ Processed successfully!", "success");

  } catch (err) {
    setStatus("error", `❌ ${err.message}`);
    showToast(`❌ ${err.message}`, "error");
  }

  hideProgress();
}

// ── Audio Player ────────────────────────────────────────────
function setupAudioPlayer(url, mimeType) {
  audioPlayer.src = url;
  audioPlayer.load();
  playBtn.disabled = false;
  playbackPanel.classList.add("visible");

  audioPlayer.onloadedmetadata = () => {
    audioTrack.max = audioPlayer.duration;
    updateTimeDisplay();
  };

  audioPlayer.ontimeupdate = () => {
    audioTrack.value = audioPlayer.currentTime;
    updateTimeDisplay();
  };

  audioPlayer.onended = () => {
    playIcon.textContent = "▶";
    playLabel.textContent = "Play";
    audioTrack.value = 0;
    updateTimeDisplay();
  };

  showToast("🔊 Recording ready — press Play to listen!", "success");
}

function togglePlay() {
  if (audioPlayer.paused) {
    audioPlayer.play();
    playIcon.textContent = "⏸";
    playLabel.textContent = "Pause";
  } else {
    audioPlayer.pause();
    playIcon.textContent = "▶";
    playLabel.textContent = "Play";
  }
}

function seekAudio(val) {
  audioPlayer.currentTime = parseFloat(val);
}

function setVolume(val) {
  audioPlayer.volume = parseFloat(val);
}

function updateTimeDisplay() {
  const cur = formatTime(audioPlayer.currentTime);
  const dur = isNaN(audioPlayer.duration) ? "0:00" : formatTime(audioPlayer.duration);
  audioTime.textContent = `${cur} / ${dur}`;
}

function formatTime(sec) {
  if (isNaN(sec)) return "0:00";
  const m = Math.floor(sec / 60);
  const s = Math.floor(sec % 60);
  return `${m}:${String(s).padStart(2, "0")}`;
}

// ── Display Results ─────────────────────────────────────────
function displayResults(data, elapsed) {
  setText(rawTextEl,    data.raw_spoken       || "—");
  setText(nepaliTextEl, data.pure_nepali      || "—");
  setText(romanTextEl,  data.romanized_nepali || "—");

  if (data.metadata) {
    const m = data.metadata;
    document.getElementById("metaLang").textContent   = (m.detected_language || "—").toUpperCase();
    document.getElementById("metaBytes").textContent  = `${(m.audio_bytes / 1024).toFixed(1)} KB`;
    document.getElementById("metaDevice").textContent = (m.processing_device || "—").toUpperCase();
    document.getElementById("metaTime").textContent   = `${elapsed}s`;
    document.getElementById("metaDict").textContent   = m.loanwords_dictionary || "—";
    metaStrip.style.display = "flex";
  }
}

function setText(el, text) {
  el.classList.remove("placeholder");
  el.textContent = text;
}

function clearResults() {
  [
    [rawTextEl,    "Verbatim speech — Nepali & English as spoken…"],
    [nepaliTextEl, "शुद्ध नेपाली देवनागरी…"],
    [romanTextEl,  "Phonetic romanization will appear here…"],
  ].forEach(([el, txt]) => {
    el.classList.add("placeholder");
    el.textContent = txt;
  });
  metaStrip.style.display = "none";
}

// ── Copy ─────────────────────────────────────────────────────
async function copyText(id) {
  const el = document.getElementById(id);
  if (!el.textContent || el.classList.contains("placeholder")) {
    showToast("Nothing to copy yet", "warning"); return;
  }
  try {
    await navigator.clipboard.writeText(el.textContent);
    showToast("📋 Copied!", "success");
  } catch {
    const ta = document.createElement("textarea");
    ta.value = el.textContent; document.body.appendChild(ta);
    ta.select(); document.execCommand("copy"); document.body.removeChild(ta);
    showToast("📋 Copied!", "success");
  }
}

// ── Toast ─────────────────────────────────────────────────────
function showToast(msg, type = "info") {
  const c = document.getElementById("toastContainer");
  const t = document.createElement("div");
  t.className = type === "success" ? "toast success-toast" : "toast";
  t.textContent = msg; c.appendChild(t);
  setTimeout(() => {
    t.style.animation = "slide-in .3s ease reverse";
    setTimeout(() => { if (c.contains(t)) c.removeChild(t); }, 300);
  }, 4000);
}

// ── Keyboard shortcut ─────────────────────────────────────────
recordBtn.addEventListener("click", toggleRecording);
document.addEventListener("keydown", e => {
  if (e.code === "Space" && e.target === document.body) {
    e.preventDefault(); toggleRecording();
  }
  // P key to play/pause
  if (e.code === "KeyP" && e.target === document.body && !playBtn.disabled) {
    togglePlay();
  }
});

// ── Health check ─────────────────────────────────────────────
window.addEventListener("load", async () => {
  setStatus("idle", "🔍 Connecting to server…");
  try {
    const res = await fetch(`${API_BASE_URL}/health`, {
      signal: AbortSignal.timeout(10000)
    });
    if (res.ok) {
      const d = await res.json();
      const gpu = d.gpu_name || "CPU";
      setStatus("ready", `✅ Online — ${gpu}`);
      showToast(`🚀 Connected! ${d.models?.loanwords || ""} loaded`, "success");
    } else throw new Error(`HTTP ${res.status}`);
  } catch (err) {
    setStatus("error", "⚠️ Server offline — check Colab");
    showToast("⚠️ Cannot reach server. Is Colab running?", "error");
  }
});
</script>
</body>
</html>'''

with open("/content/index.html", "w", encoding="utf-8") as f:
    f.write(HTML_CONTENT)

print("✅ Frontend HTML written with playback feature!")

✅ Frontend HTML written with playback feature!


In [ ]:
# ============================================================
# CELL 9: Start Ngrok + Flask
# ============================================================
import time

def update_html_with_ngrok_url(url):
    with open("/content/index.html", "r", encoding="utf-8") as f:
        html = f.read()
    html = html.replace("NGROK_URL_PLACEHOLDER", url)
    with open("/content/index.html", "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ HTML updated: {url}")

def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False,
            use_reloader=False, threaded=True)

print("🌐 Starting Ngrok...")
ngrok.kill()
time.sleep(1)

tunnel     = ngrok.connect(5000, "http")
public_url = tunnel.public_url

print(f"\n{'='*55}")
print(f"🔗 URL:      {public_url}")
print(f"🌐 Frontend: {public_url}/")
print(f"📡 Health:   {public_url}/health")
print(f"{'='*55}\n")

update_html_with_ngrok_url(public_url)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)

print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅ SYSTEM ONLINE                                     ║
║                                                      ║
║  🌐 {public_url:<48}║
║                                                      ║
║  What's new:                                         ║
║  📚 500+ loanword dictionary                         ║
║  🔒 Triple-pass loanword protection                  ║
║  🔧 Wrong-translation fix table                      ║
║  🔊 Recording playback (press P or click Play)       ║
║                                                      ║
║  Loanword examples:                                  ║
║  help       → हेल्प   (not मद्दत)                    ║
║  emergency  → इमर्जेन्सी (not आपतकाल)               ║
║  balance    → ब्यालेन्स (not शेष)                    ║
║  account    → अकाउन्ट  (not खाता)                    ║
╚══════════════════════════════════════════════════════╝
""")

while True:
    time.sleep(60)
    print(f"[{time.strftime('%H:%M:%S')}] ✅ Running — {public_url}")

🌐 Starting Ngrok...

🔗 URL:      https://designer-scrabble-calzone.ngrok-free.dev
🌐 Frontend: https://designer-scrabble-calzone.ngrok-free.dev/
📡 Health:   https://designer-scrabble-calzone.ngrok-free.dev/health

✅ HTML updated: https://designer-scrabble-calzone.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.



╔══════════════════════════════════════════════════════╗
║  ✅ SYSTEM ONLINE                                     ║
║                                                      ║
║  🌐 https://designer-scrabble-calzone.ngrok-free.dev║
║                                                      ║
║  What's new:                                         ║
║  📚 500+ loanword dictionary                         ║
║  🔒 Triple-pass loanword protection                  ║
║  🔧 Wrong-translation fix table                      ║
║  🔊 Recording playback (press P or click Play)       ║
║                                                      ║
║  Loanword examples:                                  ║
║  help       → हेल्प   (not मद्दत)                    ║
║  emergency  → इमर्जेन्सी (not आपतकाल)               ║
║  balance    → ब्यालेन्स (not शेष)                    ║
║  account    → अकाउन्ट  (not खाता)                    ║
╚══════════════════════════════════════════════════════╝



INFO:werkzeug:127.0.0.1 - - [24/Apr/2026 07:29:15] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Apr/2026 07:29:16] "GET /health HTTP/1.1" 200 -


[07:30:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev


...

[07:56:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev


INFO:werkzeug:127.0.0.1 - - [24/Apr/2026 07:56:23] "POST /process-audio HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Apr/2026 07:56:43] "POST /process-audio HTTP/1.1" 200 -


[07:57:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[07:58:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[07:59:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:00:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:01:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:02:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:03:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:04:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:05:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:06:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:07:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:08:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:09:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:10:07] ✅ Running — https://designer-scrabble-calzone.ngrok-f

INFO:werkzeug:127.0.0.1 - - [24/Apr/2026 08:35:33] "GET / HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [24/Apr/2026 08:35:34] "GET /health HTTP/1.1" 200 -


[08:36:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:37:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:38:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:39:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:40:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:41:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:42:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:43:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:44:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:45:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:46:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:47:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:48:07] ✅ Running — https://designer-scrabble-calzone.ngrok-free.dev
[08:49:07] ✅ Running — https://designer-scrabble-calzone.ngrok-f